In [172]:
import pandas as pd
import numpy as np
from nltk.corpus import stopwords
import string
import nltk
from sklearn.feature_extraction.text import TfidfVectorizer
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/anvitmangal/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [39]:
data = pd.read_csv('output_meta_yelpResData_NRYRcleaned.csv', names = ['date', 'review_id', 'reviewer_id', 'business_id', 'label', 'usefull', 'funny', 'cool', 'Stars'])

In [40]:
len(data.index)

61541

In [41]:
text_review = pd.read_csv('output_review_yelpResData_NRYRcleaned.csv', names = ['review'])

In [42]:
len(text_review.index)

61541

In [43]:
data['date'] =  pd.to_datetime(data['date'])

In [47]:
full_review = pd.concat([data, text_review], axis=1)

In [48]:
full_review

,date,review_id,reviewer_id,business_id,label,usefull,funny,cool,Stars,review
0,2012-09-22,GtwU21YOQn-wf4vWRUIx6w,bNYesZ944s6IJVowOnB0iA,pbEiXam9YJL3neCYHGwLUA,N,0,0,0,5,"Unlike Next, which we'd eaten at the previous ..."
1,2012-09-22,0LpVTc3,TRKxLC3y-ZvP45e5iilMtw,pbEiXam9YJL3neCYHGwLUA,N,0,0,0,5,Probably one of the best meals I've had ever. ...
2,2012-09-19,tljtLzf68Fkwf,0EMm8umAqXZzyhxNpL4M9g,pbEiXam9YJL3neCYHGwLUA,N,0,0,2,3,Service was impeccable. Experience and present...
3,2012-09-06,iSN,DlwexC7z88ymAzu45skODw,pbEiXam9YJL3neCYHGwLUA,N,3,0,8,3,"The problem with places like this, given the e..."
4,2012-09-09,Jmwrh7,kW2dk1CWihmh3g7k9N2G8A,pbEiXam9YJL3neCYHGwLUA,N,0,2,1,5,I have no idea how to write my review - dining...
5,2012-08-30,lKlceLWoePzeuvFD3sj4mw,HxXEcMDDTJFUqVfhPF9M8Q,pbEiXam9YJL3neCYHGwLUA,N,1,1,3,5,Despite the first-world tragedy I endured in a...
6,2012-09-08,PBS2uyee9V5IpFfTropxbw,OW2H-GkKnlVEBPuGHIaiFg,pbEiXam9YJL3neCYHGwLUA,N,0,0,1,5,"Overall, was it worth the hype? Yes and more. ..."
7,2012-08-24,PkwbB,BSh3h1J4mdSmEsb8FFdf0Q,pbEiXam9YJL3neCYHGwLUA,N,0,0,8,3,There are already TONS of professional & amate...
8,2012-08-22,MRdliMXsmP1ViLjA5oCO,F3mbveXX30Ou0gpDY6IrCQ,pbEiXam9YJL3neCYHGwLUA,N,3,7,3,5,Your life is a countdown ever since you're bor...
9,2012-08-23,z-j4X,NvSnBp4fTpNOfDwm2GWusA,pbEiXam9YJL3neCYHGwLUA,N,0,0,3,5,Lots of complaints about how difficult it is t...


In [49]:
full_review.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 61541 entries, 0 to 61540
Data columns (total 10 columns):
date           61541 non-null datetime64[ns]
review_id      61541 non-null object
reviewer_id    61541 non-null object
business_id    61541 non-null object
label          61541 non-null object
usefull        61541 non-null int64
funny          61541 non-null int64
cool           61541 non-null int64
Stars          61541 non-null int64
review         61541 non-null object
dtypes: datetime64[ns](1), int64(4), object(5)
memory usage: 4.7+ MB


In [50]:
full_review.describe()

,usefull,funny,cool,Stars
count,61541.000000,61541.000000,61541.000000,61541.000000
mean,0.453584,0.529468,0.819031,3.976633
std,1.628040,1.671220,1.897604,1.109216
min,0.000000,0.000000,0.000000,1.000000
25%,0.000000,0.000000,0.000000,4.000000
50%,0.000000,0.000000,0.000000,4.000000
75%,0.000000,1.000000,1.000000,5.000000
max,78.000000,170.000000,84.000000,5.000000


In [52]:
def text_process(review):
    """
    Takes in a string of text, then performs the following:
    1. Remove all punctuation
    2. Remove all stopwords
    3. Returns a list of the cleaned text
    """
    # Check characters to see if they are in punctuation
    nopunc = [char for char in review if char not in string.punctuation]

    # Join the characters again to form the string.
    nopunc = ''.join(nopunc)
    
    # remove any stopwords
    return [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]

In [55]:
p_reviews=full_review['review'].apply(text_process)

In [57]:
len_reviews=[len(rev) for rev in p_reviews]

In [58]:
full_review['p_reviews']=p_reviews
full_review['len_reviews']=len_reviews

In [61]:
full_review['extreme_rating']=full_review['Stars'].apply(lambda x: 1 if x==1 or x==5 else 0)

In [82]:
review_count=full_review.groupby('business_id',sort=False).count()['Stars']
review_mean=full_review.groupby('business_id',sort=False).mean()['Stars']
mean_rating=[]
for i in range(review_mean.shape[0]):
    mean_rating.extend([review_mean[i]] * review_count[i])
full_review['mean_rating']=mean_rating

In [147]:
full_review['rating_deviation']=np.abs(full_review['Stars']-full_review['mean_rating'])/4

In [108]:
temp=full_review.groupby(['reviewer_id','date'],).count()['review_id']
temp_index=temp.index

In [135]:
maxrevs=[]
reviewers=[temp_index[0][0]]
temp2=[temp[0]]
for i in range(1,len(temp)):
    if(temp_index[i][0]==temp_index[i-1][0]):
        temp2.append(temp[i])
    else:
        maxrevs.append(np.max(temp2))
        reviewers.append(temp_index[i][0])
        temp2=[temp[i]]
maxrevs.append(np.max(temp2))

In [136]:
maxrev=full_review.groupby(['reviewer_id','date']).count()['review_id'].max()

In [137]:
maxrevs=np.array(maxrevs/maxrev)

In [142]:
MNR=[]
for i in range(full_review.shape[0]):
    ind=reviewers.index(full_review.reviewer_id[i])
    MNR.append(maxrevs[ind])

In [146]:
full_review['MNR']=MNR

In [167]:
unique_revs=full_review['reviewer_id'].unique()

In [168]:
extreme_ratio=[]
for i in range(len(unique_revs)):
    if(i%100==0):
        print(i)
    df=full_review[full_review['reviewer_id']==unique_revs[i]]
    t_count=df.shape[0]
    ex_count=np.sum(df['extreme_rating'])
    extreme_ratio.append(ex_count/t_count)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200
18300
18400
18

In [169]:
ex_ratio=[]
for i in range(full_review.shape[0]):
    ind=list(unique_revs).index(full_review.reviewer_id[i])
    ex_ratio.append(extreme_ratio[ind])

In [170]:
full_review['extreme_ratio']=ex_ratio

In [173]:
vect = TfidfVectorizer(min_df=1)

In [184]:
cosin=[]
for i in range(len(unique_revs)):
    if(i%100==0):
        print(i)
    df=full_review[full_review['reviewer_id']==unique_revs[i]]
    revs=df['p_reviews'].apply(lambda x: ' '.join(x))
    max_value=0
    if(revs.values[0]!=''):
        tfidf = vect.fit_transform(revs)
        cosin_mat=(tfidf * tfidf.T).A
        np.fill_diagonal(cosin_mat, -np.inf)
        max_value = cosin_mat.max()
    cosin.append(max_value)

0
100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
12000
12100
12200
12300
12400
12500
12600
12700
12800
12900
13000
13100
13200
13300
13400
13500
13600
13700
13800
13900
14000
14100
14200
14300
14400
14500
14600
14700
14800
14900
15000
15100
15200
15300
15400
15500
15600
15700
15800
15900
16000
16100
16200
16300
16400
16500
16600
16700
16800
16900
17000
17100
17200
17300
17400
17500
17600
17700
17800
17900
18000
18100
18200
18300
18400
18

In [185]:
MCS=[]
for i in range(full_review.shape[0]):
    ind=list(unique_revs).index(full_review.reviewer_id[i])
    MCS.append(cosin[ind])

In [186]:
full_review['MCS']=MCS

In [187]:
full_review

,date,review_id,reviewer_id,business_id,label,usefull,funny,cool,Stars,review,p_reviews,len_reviews,extreme_rating,rating_deviation,mean_rating,MNR,extreme_ratio,MCS
0,2012-09-22,GtwU21YOQn-wf4vWRUIx6w,bNYesZ944s6IJVowOnB0iA,pbEiXam9YJL3neCYHGwLUA,N,0,0,0,5,"Unlike Next, which we'd eaten at the previous ...","[Unlike, Next, wed, eaten, previous, night, di...",450,1,0.099206,4.603175,0.055556,0.500000,0.107020
1,2012-09-22,0LpVTc3,TRKxLC3y-ZvP45e5iilMtw,pbEiXam9YJL3neCYHGwLUA,N,0,0,0,5,Probably one of the best meals I've had ever. ...,"[Probably, one, best, meals, Ive, ever, perfor...",34,1,0.099206,4.603175,0.055556,1.000000,-inf
2,2012-09-19,tljtLzf68Fkwf,0EMm8umAqXZzyhxNpL4M9g,pbEiXam9YJL3neCYHGwLUA,N,0,0,2,3,Service was impeccable. Experience and present...,"[Service, impeccable, Experience, presentation...",25,0,0.400794,4.603175,0.055556,0.000000,-inf
3,2012-09-06,iSN,DlwexC7z88ymAzu45skODw,pbEiXam9YJL3neCYHGwLUA,N,3,0,8,3,"The problem with places like this, given the e...","[problem, places, like, given, exhorbitant, co...",206,0,0.400794,4.603175,0.055556,0.000000,-inf
4,2012-09-09,Jmwrh7,kW2dk1CWihmh3g7k9N2G8A,pbEiXam9YJL3neCYHGwLUA,N,0,2,1,5,I have no idea how to write my review - dining...,"[idea, write, review, dining, Alinea, brings, ...",82,1,0.099206,4.603175,0.055556,0.363636,0.201236
5,2012-08-30,lKlceLWoePzeuvFD3sj4mw,HxXEcMDDTJFUqVfhPF9M8Q,pbEiXam9YJL3neCYHGwLUA,N,1,1,3,5,Despite the first-world tragedy I endured in a...,"[Despite, firstworld, tragedy, endured, effort...",197,1,0.099206,4.603175,0.055556,1.000000,-inf
6,2012-09-08,PBS2uyee9V5IpFfTropxbw,OW2H-GkKnlVEBPuGHIaiFg,pbEiXam9YJL3neCYHGwLUA,N,0,0,1,5,"Overall, was it worth the hype? Yes and more. ...","[Overall, worth, hype, Yes, worth, every, penn...",236,1,0.099206,4.603175,0.055556,1.000000,-inf
7,2012-08-24,PkwbB,BSh3h1J4mdSmEsb8FFdf0Q,pbEiXam9YJL3neCYHGwLUA,N,0,0,8,3,There are already TONS of professional & amate...,"[already, TONS, professional, amateur, reviews...",230,0,0.400794,4.603175,0.111111,0.250000,0.150627
8,2012-08-22,MRdliMXsmP1ViLjA5oCO,F3mbveXX30Ou0gpDY6IrCQ,pbEiXam9YJL3neCYHGwLUA,N,3,7,3,5,Your life is a countdown ever since you're bor...,"[life, countdown, ever, since, youre, born, li...",265,1,0.099206,4.603175,0.111111,0.538462,0.235199
9,2012-08-23,z-j4X,NvSnBp4fTpNOfDwm2GWusA,pbEiXam9YJL3neCYHGwLUA,N,0,0,3,5,Lots of complaints about how difficult it is t...,"[Lots, complaints, difficult, get, get, hard, ...",70,1,0.099206,4.603175,0.055556,1.000000,-inf


In [189]:
full_review.to_csv('Extracted_features.csv')

In [188]:
print(full_review.groupby('label').describe())

           MCS                                                  MNR            \
         count mean std  min  25%       50%       75%  max    count      mean   
label                                                                           
N      53400.0 -inf NaN -inf -inf  0.074927  0.143081  1.0  53400.0  0.082222   
Y       8141.0 -inf NaN -inf  NaN       NaN  0.000000  1.0   8141.0  0.069607   

       ...  rating_deviation            usefull                               \
       ...               75%       max    count      mean      std  min  25%   
label  ...                                                                     
N      ...          0.270130  0.900794  53400.0  0.522734  1.73737  0.0  0.0   
Y      ...          0.384558  0.746959   8141.0  0.000000  0.00000  0.0  0.0   

                       
       50%  75%   max  
label                  
N      0.0  0.0  78.0  
Y      0.0  0.0   0.0  

[2 rows x 88 columns]


/Users/anvitmangal/miniconda3/lib/python3.6/site-packages/numpy/lib/function_base.py:4406: RuntimeWarning: invalid value encountered in multiply
  x2 = take(ap, indices_above, axis=axis) * weights_above
